# Inflection analysis for Annual & Monthly Return Period Analysis Pipeline

### Description:
This Python workflow ingests multi‑year ADCIRC+SWAN hindcast outputs to:

Compute Annual Maxima & Inflection Years:

Load yearly water‑level datasets.

Derive each node’s annual maximum water surface elevation.

Detect a single annual change‑point (“inflection year”) per node using a binary segmentation algorithm.

Map and tabulate the most common inflection years on a North‑Polar‑Stereo plot.

Calculate Monthly Return Levels Around Inflection:

Resample the full time series into monthly maxima.

For each node, split its monthly series at the detected inflection year into pre‑ and post‑periods.

Fit a Gumbel distribution to each segment and estimate T‑year return levels (e.g., 10, 50, 100, 500 years).

Visualize Results:

Annual: Polar scatter of top inflection years with a discrete colorbar.

Monthly: A 6×6 grid of polar maps showing pre‑ vs. post‑inflection return levels by month, plus the percent change (ΔT), all annotated with month labels, panel IDs, and shared colorbars.

### Initialize Libraries

In [ ]:
import warnings;            import os
import pathlib as pl;       import string
import calendar;            import numpy as np
import pandas as pd;        import xarray as xr
import ruptures as rpt;     import matplotlib.pyplot as plt
import matplotlib as mpl;   import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask.distributed import Client, LocalCluster
from scipy.stats import pearsonr, genextreme, gumbel_r
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import HypothesisTestWarning
warnings.filterwarnings("ignore")

In [ ]:
def compute_keep_idx(sample_path, min_depth, max_depth, bbox=None):
    """
    Return indices of nodes whose depth ∈ [min_depth, max_depth],
    optionally also within (min_x, max_x, min_y, max_y).
    """
    ds = xr.open_dataset(sample_path, engine="netcdf4", mask_and_scale=False)
    depth = ds.depth.values
    fv    = ds.depth.encoding.get("_FillValue", None)

    mask = np.ones_like(depth, dtype=bool)
    if fv is not None:
        mask &= (depth != fv)
    mask &= (depth >= min_depth) & (depth <= max_depth)

    if bbox is not None:
        min_x, max_x, min_y, max_y = bbox
        x, y = ds.x.values, ds.y.values
        mask &= (x >= min_x) & (x <= max_x) & (y >= min_y) & (y <= max_y)

    ds.close()
    return np.where(mask)[0]


def preprocess(ds):
    """Keep only the spin‑up‑free calendar year & shallow nodes."""
    ds = ds.assign_coords(time=pd.to_datetime(ds.time.values))
    year = ds.time.dt.year.values[600]
    mask = ds.time.dt.year == year
    return ds.isel(time=mask, node=keep_idx)


# ─── Change-point detection function ──────────────────────────────────────────
def detect_cp_annual(y, years_array, n_bkps=1):
    """
    Identify one change-point in a 1D annual series using ruptures.
    Returns the corresponding year or NaN if no valid break.
    """
    # need at least 5 valid points
    if np.sum(~np.isnan(y)) < 5:
        return np.nan

    algo = rpt.Binseg(model='rbf').fit(y.reshape(-1, 1))
    bkps = algo.predict(n_bkps=n_bkps)  # e.g., [idx]
    cp   = bkps[0]

    # ensure break is interior
    if 0 < cp < len(years_array):
        return years_array[cp]
    return np.nan

def test_stationarity(arr, alpha=0.05):
    """
    Returns (is_stationary, p_adf, p_kpss) for a 1D array.
    ADF stationary if p_adf < alpha; KPSS stationary if p_kpss > alpha.
    """
    # ADF
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", HypothesisTestWarning)
        try:
            p_adf = adfuller(arr, autolag='AIC')[1]
        except Exception:
            p_adf = 0.0
    # KPSS
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", HypothesisTestWarning)
        try:
            p_kpss = kpss(arr, regression='c', nlags='auto')[1]
        except Exception:
            p_kpss = 1.0

    p_adf  = min(max(p_adf, 0.0), 1.0)
    p_kpss = min(max(p_kpss, 0.0), 1.0)
    return (p_adf < alpha) and (p_kpss > alpha), p_adf, p_kpss


def make_labels(n):
    """Generate ‘a.’, ‘b.’ … ‘z.’, ‘aa.’, … for n panels."""
    labels = []
    alphabet = string.ascii_lowercase
    i = 0
    while len(labels) < n:
        s, x = '', i
        while True:
            s = alphabet[x % 26] + s
            x = x // 26 - 1
            if x < 0:
                break
        labels.append(s + '.')
        i += 1
    return labels


In [ ]:
# ─── User‑set parameters ──────────────────────────────────────────────────────
lat1, lat2 = 56, 75
lon1, lon2 = -168.5, -140
root       = '/scratch/tmiesse/project/data4spatial'
YEARS      = range(1980, 2025)
MIN_DEPTH  = 1.0
MAX_DEPTH  = 10
bbox       = (lon1, lon2, lat1, lat2)

# compute shallow nodes index
sample_path = f"{root}/2023/fort.63.cf.nc"
keep_idx    = compute_keep_idx(sample_path, MIN_DEPTH, MAX_DEPTH, bbox)
print(f"Keeping {len(keep_idx)} nodes")

In [ ]:

# ─── Spin up a Dask cluster ───────────────────────────────────────────────────
n_workers   = int(os.environ.get("SLURM_NTASKS", "45"))
mem_per_cpu = os.environ.get("SLURM_MEM_PER_CPU", "72GB")
cluster     = LocalCluster(
    n_workers=n_workers,
    threads_per_worker=1,
    processes=True,
    memory_limit=mem_per_cpu
)
client = Client(cluster)
print(client)

In [ ]:
paths = [str(pl.Path(root)/str(y)/"swan_HS.63.cf.nc") for y in YEARS]
ds_all = xr.open_mfdataset(
    paths,
    engine="h5netcdf",
    mask_and_scale=True,
    decode_cf=True,
    preprocess=preprocess,
    combine="nested",
    concat_dim="time",
    data_vars=['swan_HS'],               # only load zeta
    chunks={'time': 24},# chunk on both dims
    coords="minimal"                  # minimal coords → time, node, x, y
)

times  = ds_all.time.values      # length T
hs = ds_all.swan_HS.values
#z = ds_all.zeta.values.astype(float)
x_all  = ds_all.x.values                          # (n_nodes,)
y_all  = ds_all.y.values                          # (n_nodes,)


In [ ]:


bins_pp    = np.linspace(0, 3.5, 8)
cmap_pp    = plt.get_cmap('rainbow', len(bins_pp)-1)
norm_pp    = mpl.colors.BoundaryNorm(bins_pp, len(bins_pp)-1)

fig = plt.figure(figsize=(7, 5))
ax  = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude=-145))
ax.set_extent([-178, -140, 58, 78], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND,  facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
ax.coastlines(resolution='10m', linewidth=0.5)
ax.gridlines(
    xlocs=np.arange(-180, -100, 5),
    ylocs=np.arange(50, 90, 3),
    draw_labels=False, linewidth=0.2,
    color='black', alpha=0.3, linestyle='--'
)
arr = np.mean(z[300:1280,:], axis=0)  # average over the years
arr[arr<=0.01] = np.nan
sc = ax.scatter(
    x_all, y_all,c=arr,
    cmap=cmap_pp, norm=norm_pp,
    s=2.5, transform=ccrs.PlateCarree(), zorder=2
)
cax = fig.add_axes([0.85, 0.11, 0.02, 0.77])
cb  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_pp, cmap=cmap_pp),
    cax=cax, orientation='vertical',
    boundaries=bins_pp, ticks=np.arange(len(bins_pp)-1)
)
cb.set_ticklabels(np.arange(len(bins_pp)-1))
cb.set_label('Hs[m]')

plt.show()

In [ ]:
# ─── Load & concatenate annual zeta ────────────────────────────────────────────
paths = [str(pl.Path(root)/str(y)/"swan_HS.63.cf.nc") for y in YEARS]
ds_all = xr.open_mfdataset(
    paths,
    engine="h5netcdf",
    mask_and_scale=True,
    decode_cf=True,
    preprocess=preprocess,
    combine="nested",
    concat_dim="time",
    chunks={'time': 24}
)
z       = ds_all.swan_HS.values
x_all   = ds_all.x.values
y_all   = ds_all.y.values

years_all    = ds_all.time.dt.year.values
years_unique = np.unique(years_all)
nnode        = z.shape[1]

In [ ]:
# compute annual maxima
annual_max = np.full((len(years_unique), nnode), np.nan)
for i, yr in enumerate(years_unique):
    mask = years_all == yr
    if mask.any():
        annual_max[i, :] = np.nanmax(z[mask, :], axis=0)

da_annual = xr.DataArray(
    annual_max,
    dims=('year','node'),
    coords={
        'year': years_unique,
        'node': ds_all.node,
        'x':     ('node', x_all),
        'y':     ('node', y_all),
    },
    name='annual_max_swan_HS'
)

In [ ]:
# ─── Apply change-point detection across nodes ────────────────────────────────
cp_da = xr.apply_ufunc(
    detect_cp_annual,
    da_annual,
    input_core_dims=[['year']],
    output_core_dims=[[]],
    vectorize=True,
    kwargs={'years_array': da_annual.year.values},
    dask='parallelized',
    output_dtypes=[float]
).rename('inflection_year')

# reattach spatial coordinates
cp_da = cp_da.assign_coords({
    'x': ('node', x_all),
    'y': ('node', y_all),
})

# ─── Save to NetCDF ───────────────────────────────────────────────────────────
ds_inf = cp_da.to_dataset()
outpath = '/scratch/tmiesse/project/inflection_annual_hs.nc'
ds_inf.to_netcdf(outpath)

In [ ]:
# ─── Plot top‑4 inflection years on a polar map ───────────────────────────────
inf_years = ds_inf.inflection_year.values
lon, lat  = ds_inf.x.values, ds_inf.y.values

mask = ~np.isnan(inf_years)
yrs, cnts = np.unique(inf_years[mask].astype(int), return_counts=True)
top4     = np.sort(yrs[np.argsort(cnts)[-4:]])
bin_idx  = np.full_like(inf_years, -1, int)
for b, yr in enumerate(top4):
    bin_idx[inf_years == yr] = b
mask4 = bin_idx >= 0

cmap    = plt.get_cmap('tab10', len(top4))
bounds  = np.arange(len(top4)+1) - 0.5
norm    = mpl.colors.BoundaryNorm(bounds, len(top4))

fig = plt.figure(figsize=(7, 5))
ax  = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude=-145))
ax.set_extent([-168, -140, 58, 71], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND,  facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
ax.coastlines(resolution='10m', linewidth=0.5)
ax.gridlines(
    xlocs=np.arange(-180, -100, 5),
    ylocs=np.arange(50, 90, 3),
    draw_labels=False, linewidth=0.2,
    color='black', alpha=0.3, linestyle='--'
)
sc = ax.scatter(
    lon[mask4], lat[mask4],
    c=bin_idx[mask4], cmap=cmap, norm=norm,
    s=2.5, transform=ccrs.PlateCarree(), zorder=2
)
cax = fig.add_axes([0.85, 0.11, 0.02, 0.77])
cb  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=cax, orientation='vertical',
    boundaries=bounds, ticks=np.arange(len(top4))
)
cb.set_ticklabels(top4.astype(int))
cb.set_label('Inflection Year')
plt.savefig(
    '/scratch/tmiesse/project/figures/inflection_map_hs.png',
    dpi=720, bbox_inches='tight', pad_inches=0.1
)
plt.show()

In [ ]:
# ─── Stationarity tests (placeholders) ────────────────────────────────────────
# NOTE: arrays pre_stat, post_stat, pre_p_adf, etc. are initialized below—
#       fill them using test_stationarity before saving if needed.
years_arr = da_annual.year.values.astype(int)
nnode     = da_annual.sizes['node']

pre_stat   = np.full(nnode, False, dtype=bool)
post_stat  = np.full(nnode, False, dtype=bool)
pre_p_adf  = np.full(nnode, np.nan)
pre_p_kpss = np.full(nnode, np.nan)
post_p_adf = np.full(nnode, np.nan)
post_p_kpss= np.full(nnode, np.nan)

ds_stat = xr.Dataset({
    'pre_stationary':   (('node',), pre_stat),
    'post_stationary':  (('node',), post_stat),
    'pre_p_adf':        (('node',), pre_p_adf),
    'pre_p_kpss':       (('node',), pre_p_kpss),
    'post_p_adf':       (('node',), post_p_adf),
    'post_p_kpss':      (('node',), post_p_kpss),
}, coords={'x':('node', x_all), 'y':('node', y_all)})

ds_stat.to_netcdf('/scratch/tmiesse/project/inflection_stationarity_hs.nc')

In [ ]:
# === Parameters ===
alpha = 0.05  # significance level
min_samples = 5  # minimum non-NaN points to test
years_arr = da_annual.year.values.astype(int)  # e.g. [1980,...]
nyears = len(years_arr)
nnode = da_annual.sizes['node']
inf_years = xr.open_dataset('/scratch/tmiesse/project/inflection_annual_hs.nc')['inflection_year'].values

# === Containers ===
pre_stat_strict   = np.full(nnode, False, dtype=bool)
post_stat_strict  = np.full(nnode, False, dtype=bool)
pre_p_adf         = np.full(nnode, np.nan)
pre_p_kpss        = np.full(nnode, np.nan)
post_p_adf        = np.full(nnode, np.nan)
post_p_kpss       = np.full(nnode, np.nan)

pre_stat_loose    = np.full(nnode, False, dtype=bool)
post_stat_loose   = np.full(nnode, False, dtype=bool)

# === Helper test function ===
def compute_adf_kpss(arr):
    """Return (p_adf, p_kpss), with safe fallbacks."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", HypothesisTestWarning)
        try:
            p_adf = adfuller(arr, autolag='AIC')[1]
        except Exception:
            p_adf = 1.0  # fail conservative
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", HypothesisTestWarning)
        try:
            p_kpss = kpss(arr, regression='c', nlags='auto')[1]
        except Exception:
            p_kpss = 0.0  # fail conservative
    # clamp
    p_adf  = min(max(p_adf, 0.0), 1.0)
    p_kpss = min(max(p_kpss, 0.0), 1.0)
    return p_adf, p_kpss

# === Year to index map for inflection split ===
year_to_idx = {int(yr): idx for idx, yr in enumerate(years_arr)}

# === Loop over nodes ===
for i in range(nnode):
    cp_year = inf_years[i]
    if np.isnan(cp_year):
        continue
    cp_year = int(cp_year)
    cp_idx = year_to_idx.get(cp_year, None)
    if cp_idx is None or cp_idx == 0 or cp_idx >= nyears:
        continue

    # Pre-inflection segment
    pre_series = da_annual.values[:cp_idx, i]
    valid_pre = pre_series[np.isfinite(pre_series)]
    if valid_pre.size >= min_samples:
        p_adf_pre, p_kpss_pre = compute_adf_kpss(valid_pre)
        pre_p_adf[i] = p_adf_pre
        pre_p_kpss[i] = p_kpss_pre
        strict_pre = (p_adf_pre < alpha) and (p_kpss_pre > alpha)
        loose_pre  = (p_adf_pre < alpha) or (p_kpss_pre > alpha)
        pre_stat_strict[i] = strict_pre
        pre_stat_loose[i]  = loose_pre

    # Post-inflection segment (includes inflection year)
    post_series = da_annual.values[cp_idx:, i]
    valid_post = post_series[np.isfinite(post_series)]
    if valid_post.size >= min_samples:
        p_adf_post, p_kpss_post = compute_adf_kpss(valid_post)
        post_p_adf[i] = p_adf_post
        post_p_kpss[i] = p_kpss_post
        strict_post = (p_adf_post < alpha) and (p_kpss_post > alpha)
        loose_post  = (p_adf_post < alpha) or (p_kpss_post > alpha)
        post_stat_strict[i] = strict_post
        post_stat_loose[i]  = loose_post

# === Decide which mask to use: strict unless it yields zero hits ===
use_loose_pre = pre_stat_strict.sum() == 0
use_loose_post = post_stat_strict.sum() == 0

pre_stat_final  = pre_stat_loose if use_loose_pre else pre_stat_strict
post_stat_final = post_stat_loose if use_loose_post else post_stat_strict

# === Diagnostics ===
print("Pre-inflection stationarity (strict) count:", np.sum(pre_stat_strict))
print("Post-inflection stationarity (strict) count:", np.sum(post_stat_strict))
if use_loose_pre:
    print("Falling back to loose pre-inflection criterion; count:", np.sum(pre_stat_loose))
if use_loose_post:
    print("Falling back to loose post-inflection criterion; count:", np.sum(post_stat_loose))

# === Build and save dataset ===
ds_stat = xr.Dataset({
    'pre_stationary':   (('node',), pre_stat_final,  {'long_name': 'pre-inflection stationary?'}),
    'post_stationary':  (('node',), post_stat_final, {'long_name': 'post-inflection stationary?'}),
    'pre_p_adf':        (('node',), pre_p_adf,       {'long_name': 'ADF p-value (pre)'}),
    'pre_p_kpss':       (('node',), pre_p_kpss,      {'long_name': 'KPSS p-value (pre)'}),
    'post_p_adf':       (('node',), post_p_adf,      {'long_name': 'ADF p-value (post)'}),
    'post_p_kpss':      (('node',), post_p_kpss,     {'long_name': 'KPSS p-value (post)'}),
}, coords={
    'node': da_annual['node'].values,
    'x': ('node', x_all),
    'y': ('node', y_all),
})

out_stat_path = '/scratch/tmiesse/project/inflection_stationarity_hs.nc'
ds_stat.to_netcdf(out_stat_path)
print(f"Saved updated stationarity file to {out_stat_path}")

In [ ]:
# ─── Parameters & prerequisites ───────────────────────────────────────────────
# Assumes YEARS, ds_all, z, x_all, y_all, inf_years are already defined
years_unique = np.array(list(YEARS))       # e.g. [1980, …, 2024]
nyears, nnode = len(years_unique), z.shape[1]

# ─── 1) Compute monthly maxima per node ───────────────────────────────────────
monthly_max = np.full((nyears, 12, nnode), np.nan)
for i, yr in enumerate(years_unique):
    mask_year = ds_all.time.dt.year == yr
    for m in range(1, 13):
        mask_month = mask_year & (ds_all.time.dt.month == m)
        if mask_month.any():
            monthly_max[i, m-1, :] = np.nanmax(z[mask_month, :], axis=0)

da_monthly = xr.DataArray(
    monthly_max,
    dims=('year','month','node'),
    coords={
        'year':  years_unique,
        'month': np.arange(1, 13),
        'node':  ds_all.node,
        'x':     ('node', x_all),
        'y':     ('node', y_all),
    },
    name='monthly_max_swan_HS'
)

# ─── 2) Prepare containers for return levels ─────────────────────────────────
return_periods = [10, 50, 100, 500]
rp_pre  = {T: np.full((12, nnode), np.nan) for T in return_periods}
rp_post = {T: np.full((12, nnode), np.nan) for T in return_periods}

# ─── 3) Loop over nodes, split at inflection, fit Gumbel, store ppf ────────
for i in range(nnode):
    cp_year = inf_years[i]
    if np.isnan(cp_year):
        continue

    # find the index of cp_year
    try:
        cp_idx = int(np.where(years_unique == cp_year)[0][0])
    except IndexError:
        continue
    if not (0 < cp_idx < nyears):
        continue

    for m in range(12):
        # Pre-inflection
        data_pre  = da_monthly[:cp_idx, m, i].values
        valid_pre = data_pre[np.isfinite(data_pre)]
        if len(valid_pre) >= 20:
            loc, scale = gumbel_r.fit(valid_pre)
            for T in return_periods:
                rp_pre[T][m, i] = gumbel_r.ppf(1 - 1/T, loc=loc, scale=scale)

        # Post-inflection
        data_post  = da_monthly[cp_idx:, m, i].values
        valid_post = data_post[np.isfinite(data_post)]
        if len(valid_post) >= 20:
            loc2, scale2 = gumbel_r.fit(valid_post)
            for T in return_periods:
                rp_post[T][m, i] = gumbel_r.ppf(1 - 1/T, loc=loc2, scale=scale2)

# ─── 4) Wrap into an xarray Dataset and save ─────────────────────────────────
data_vars = {}
for T in return_periods:
    data_vars[f'rp_pre_{T}']  = (('month','node'), rp_pre[T],
                                {'units':'m',
                                 'long_name':f'{T}-yr return level (pre)'})
    data_vars[f'rp_post_{T}'] = (('month','node'), rp_post[T],
                                {'units':'m',
                                 'long_name':f'{T}-yr return level (post)'})

ds_monthly_rp = xr.Dataset(
    data_vars,
    coords={
        'month': np.arange(1, 13),
        'node':  da_monthly.node,
        'x':     ('node', x_all),
        'y':     ('node', y_all),
    }
)



In [ ]:
outpath = '/scratch/tmiesse/project/inflection_returnlevels_monthly_hs.nc'
ds_monthly_rp.to_netcdf(outpath)

In [ ]:
# ─── 6×6 Panel: Monthly Return Levels & ΔT ────────────────────────────────────
# 1) Load the dataset and pick a return period (e.g., 10‑yr)
ds      = xr.open_dataset('/scratch/tmiesse/project/inflection_returnlevels_monthly_hs.nc')
x, y    = ds['x'].values, ds['y'].values
T       = 10
da_pre  = ds[f'rp_pre_{T}']        # dims = (month, node)
da_post = ds[f'rp_post_{T}']
da_diff = (da_post - da_pre) / da_pre * 100

# 2) Colormap settings
bins_pp    = np.linspace(0, 3.5, 8)
cmap_pp    = plt.get_cmap('rainbow', len(bins_pp)-1)
norm_pp    = mpl.colors.BoundaryNorm(bins_pp, len(bins_pp)-1)
ticks_diff = np.arange(-20, 51, 10)
cmap_diff  = plt.get_cmap('Spectral', len(ticks_diff)-1)
norm_diff  = mpl.colors.BoundaryNorm(ticks_diff, len(ticks_diff)-1)

# 3) Panel labels & row titles
n_panels     = 36
panel_labels = make_labels(n_panels)
row_titles   = [
    'Pre-Inflection', 'Pre-Inflection',
    'Post-Inflection', 'Post-Inflection',
    'ΔT',             'ΔT'
]

# 4) Figure setup: 6 rows × 6 cols of polar maps
fig, axes = plt.subplots(
    6, 6, figsize=(18, 18),
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)},
    gridspec_kw={'wspace':0.05, 'hspace':0.05}
)

panel_idx = 0
for row in range(6):
    block = row // 2   # 0=pre, 1=post, 2=diff
    for col in range(6):
        ax = axes[row, col]
        ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND,  facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
        ax.coastlines(resolution='10m', linewidth=0.1)
        ax.gridlines(
            xlocs=np.arange(-180, -100, 5),
            ylocs=np.arange(50, 90, 3),
            draw_labels=False, linewidth=0.2,
            color='black', alpha=0.3, linestyle='--'
        )

        # determine month and data for this panel
        month = (col + 1) + 6*(row % 2)  # 1–6 if row even, 7–12 if odd
        if block == 0:
            data, cmap, norm = da_pre.sel(month=month), cmap_pp, norm_pp
        elif block == 1:
            data, cmap, norm = da_post.sel(month=month), cmap_pp, norm_pp
        else:
            data, cmap, norm = da_diff.sel(month=month), cmap_diff, norm_diff

        arr = data.values
        mask = ~np.isnan(arr)
        ax.scatter(
            x[mask], y[mask], c=arr[mask],
            cmap=cmap, norm=norm,
            s=5, transform=ccrs.PlateCarree()
        )

        # month title and panel label
        ax.set_title(calendar.month_abbr[month], fontsize=10, pad=2)
        ax.text(
            0.02, 1.05, panel_labels[panel_idx],
            transform=ax.transAxes,
            fontsize=10, fontweight='bold',
            va='top'
        )
        panel_idx += 1
        ax.set_xticks([]); ax.set_yticks([])

# 5) Row titles along left margin
for row in range(6):
    pos   = axes[row, 0].get_position()
    y_mid = 0.5 * (pos.y0 + pos.y1)
    fig.text(
        pos.x0 - 0.005, y_mid,
        row_titles[row],
        fontsize=12, fontweight='bold',
        rotation='vertical', va='center', ha='right'
    )

# 6) Shared colorbars
# Pre/Post
cax1 = fig.add_axes([0.91, 0.3725, 0.015, 0.505])
cb1  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_pp, cmap=cmap_pp),
    cax=cax1, orientation='vertical',
    boundaries=bins_pp, ticks=bins_pp
)
cb1.set_label('WSE [m at MSL]', fontsize=12)

# ΔT
cax2 = fig.add_axes([0.91, 0.113, 0.015, 0.2475])
cb2  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_diff, cmap=cmap_diff),
    cax=cax2, orientation='vertical',
    boundaries=ticks_diff, ticks=ticks_diff
)
cb2.set_label('% ΔT', fontsize=12)

plt.suptitle(f'{T}-yr Return Levels by Month (6×6)', fontsize=16, y=0.92)
plt.savefig(
    '/scratch/tmiesse/project/figures/return_period_seasonal2.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
use_stationarity = True  # set False to bypass pre/post stationary gating for debugging
min_samples = 20        # original threshold
return_periods = [10, 50, 100, 500]

# === Load inputs ===
ds_stat   = xr.open_dataset('/scratch/tmiesse/project/inflection_stationarity_hs.nc')
pre_stat  = ds_stat['pre_stationary'].values
post_stat = ds_stat['post_stationary'].values

ds_inf    = xr.open_dataset('/scratch/tmiesse/project/inflection_annual_hs.nc')
inf_years = ds_inf['inflection_year'].values  # (node,)

# da_annual, x_all, y_all, years_unique must already be in scope
years_unique = da_annual['year'].values.astype(int)
nnode = da_annual.sizes['node']
nyears = len(years_unique)

# helper: map year -> index
year_to_idx = {int(yr): idx for idx, yr in enumerate(years_unique)}

# === Prepare output arrays ===
rp_pre = {T: np.full(nnode, np.nan) for T in return_periods}
rp_post = {T: np.full(nnode, np.nan) for T in return_periods}

# === Counters for diagnostics ===
count_inf_valid = 0
count_pre_sample = 0
count_post_sample = 0
count_pre_stationary = 0
count_post_stationary = 0
count_pre_fitted = 0
count_post_fitted = 0

for i in range(nnode):
    cp_year = inf_years[i]
    if np.isnan(cp_year):
        continue
    cp_year = int(cp_year)
    cp_idx = year_to_idx.get(cp_year, None)
    if cp_idx is None or cp_idx == 0 or cp_idx >= nyears:
        continue
    count_inf_valid += 1

    # Pre-inflection slice and check
    data_pre = da_annual.values[:cp_idx, i]
    valid_pre = data_pre[np.isfinite(data_pre)]
    has_pre_sample = valid_pre.size >= min_samples
    if has_pre_sample:
        count_pre_sample += 1
    if use_stationarity and pre_stat[i]:
        count_pre_stationary += 1
    if has_pre_sample and (not use_stationarity or pre_stat[i]):
        # attempt fit
        try:
            loc, scale = gumbel_r.fit(valid_pre)
            for T in return_periods:
                rp_pre[T][i] = gumbel_r.ppf(1 - 1 / T, loc=loc, scale=scale)
            count_pre_fitted += 1
        except Exception:
            # fit failed — leave NaNs
            pass

    # Post-inflection
    data_post = da_annual.values[cp_idx:, i]
    valid_post = data_post[np.isfinite(data_post)]
    has_post_sample = valid_post.size >= min_samples
    if has_post_sample:
        count_post_sample += 1
    if use_stationarity and post_stat[i]:
        count_post_stationary += 1
    if has_post_sample and (not use_stationarity or post_stat[i]):
        try:
            loc2, scale2 = gumbel_r.fit(valid_post)
            for T in return_periods:
                rp_post[T][i] = gumbel_r.ppf(1 - 1 / T, loc=loc2, scale=scale2)
            count_post_fitted += 1
        except Exception:
            pass

# === Diagnostic summary ===
print("=== Annual return level diagnostics ===")
print(f"Nodes with valid inflection split: {count_inf_valid}/{nnode}")
print(f"Nodes with enough pre-inflection samples: {count_pre_sample}")
print(f"Nodes with enough post-inflection samples: {count_post_sample}")
if use_stationarity:
    print(f"Nodes pre-stationary flagged: {count_pre_stationary}")
    print(f"Nodes post-stationary flagged: {count_post_stationary}")
print(f"Nodes with successful pre-inflection fits: {count_pre_fitted}")
print(f"Nodes with successful post-inflection fits: {count_post_fitted}")

# === Assemble dataset ===
ds_out = xr.Dataset({
    'rp_pre_10':  (('node',), rp_pre[10],  {'units':'m','long_name':'10-yr return level (pre)'}),
    'rp_pre_50':  (('node',), rp_pre[50],  {'units':'m','long_name':'50-yr return level (pre)'}),
    'rp_pre_100': (('node',), rp_pre[100], {'units':'m','long_name':'100-yr return level (pre)'}),
    'rp_pre_500': (('node',), rp_pre[500], {'units':'m','long_name':'500-yr return level (pre)'}),
    'rp_post_10':  (('node',), rp_post[10],  {'units':'m','long_name':'10-yr return level (post)'}),
    'rp_post_50':  (('node',), rp_post[50],  {'units':'m','long_name':'50-yr return level (post)'}),
    'rp_post_100': (('node',), rp_post[100], {'units':'m','long_name':'100-yr return level (post)'}),
    'rp_post_500': (('node',), rp_post[500], {'units':'m','long_name':'500-yr return level (post)'}),
}, coords={
    'node': da_annual['node'].values,
    'x':    ('node', x_all),
    'y':    ('node', y_all),
})

# === Save ===
outpath = '/scratch/tmiesse/project/inflection_returnlevels_annual_hs.nc'
ds_out.to_netcdf(outpath, format='NETCDF4', engine='h5netcdf')
print(f"Saved annual return levels to: {outpath}")

In [ ]:
# ─── 3×3 panel maps of return levels & percent differences ───────────────────
ds_rp = xr.open_dataset(outpath)
x, y = ds_rp.x.values, ds_rp.y.values

# Prepare fields for T10, T50, T100
fields_pre = [
    (r'Pre-Inflection $T_{10}$',  ds_rp.rp_pre_10.values),
    (r'Pre-Inflection $T_{50}$',  ds_rp.rp_pre_50.values),
    (r'Pre-Inflection $T_{100}$', ds_rp.rp_pre_100.values),
]
fields_post = [
    (r'Post-Inflection $T_{10}$',  ds_rp.rp_post_10.values),
    (r'Post-Inflection $T_{50}$',  ds_rp.rp_post_50.values),
    (r'Post-Inflection $T_{100}$', ds_rp.rp_post_100.values),
]

# Compute percent differences
fields_diff = []
for (lbl_pre, arr_pre), (lbl_post, arr_post) in zip(fields_pre, fields_post):
    pct = (arr_post - arr_pre) / arr_pre * 100
    Tn = lbl_pre.split('$T_{')[-1].strip('}$')
    fields_diff.append((rf'$\Delta T_{{{Tn}}}$', pct))

all_fields = fields_pre + fields_post + fields_diff
labels     = make_labels(len(all_fields))

# Colormap setup for pre/post
bins_pp = np.linspace(0, 7, 8)
cmap_pp = plt.get_cmap('rainbow', len(bins_pp)-1)
norm_pp = mpl.colors.BoundaryNorm(bins_pp, len(bins_pp)-1)

# Colormap for percent diff
vmin, vmax = -50, 50
cmap_diff  = plt.get_cmap('coolwarm', 12)
norm_diff  = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
ticks_diff = np.linspace(vmin, vmax, 11)

# Create 3×3 map panel
fig, axes = plt.subplots(
    3, 3, figsize=(12, 12),
    subplot_kw={'projection': ccrs.NorthPolarStereo(central_longitude=-145)},
    gridspec_kw={
        'left':   0.02, 'right':  0.78,
        'top':    0.99, 'bottom': 0.01,
        'wspace': 0.01, 'hspace': -0.4,
    }
)

for idx, ax in enumerate(axes.flat):
    title, data = all_fields[idx]
    ax.set_extent([-169, -140, 58, 71], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,  facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
    ax.coastlines(resolution='10m', linewidth=0.1)
    ax.gridlines(
        xlocs=np.arange(-180, -100, 5),
        ylocs=np.arange( 50,   90, 3),
        draw_labels=False, linewidth=0.2,
        color='black', alpha=0.3, linestyle='--'
    )

    mask = ~np.isnan(data)
    cmap, norm = (cmap_pp, norm_pp) if idx < 6 else (cmap_diff, norm_diff)
    ax.scatter(
        x[mask], y[mask],
        c=data[mask], cmap=cmap, norm=norm,
        s=5, transform=ccrs.PlateCarree()
    )

    ax.set_title(title, fontsize=11, pad=2)
    ax.text(
        0.02, 1.05, labels[idx],
        transform=ax.transAxes,
        fontsize=10, va='top'
    )

# Shared colorbar for pre/post rows
cax1 = fig.add_axes([0.80, 0.385, 0.015, 0.50])
cb1  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_pp, cmap=cmap_pp),
    cax=cax1, orientation='vertical',
    boundaries=bins_pp, ticks=bins_pp
)
cb1.set_label('Hs [m]', fontsize=12)

# Shared colorbar for percent‑diff row
cax2 = fig.add_axes([0.80, 0.115, 0.015, 0.235])
cb2  = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm_diff, cmap=cmap_diff),
    cax=cax2, orientation='vertical',
    ticks=ticks_diff
)
cb2.set_ticklabels([f"{t:.0f}%" for t in ticks_diff])
cb2.set_label(r'$\Delta T$ [%]', fontsize=12)

plt.savefig(
    '/scratch/tmiesse/project/figures/return_period_hs.png',
    dpi=720, bbox_inches='tight', pad_inches=0.1
)
plt.show()